# Spanish Stance Detection — Three-LM Variant
Catalonia Independence Corpus (CIC-ES). This notebook applies a **three-LM argmin** design: one GPT-2 per class (AGAINST, FAVOR, NEUTRAL). A tweet is assigned to whichever model assigns it the lowest perplexity.



## 1. Setup — Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers torch scikit-learn seaborn

In [ ]:
import random, os
import numpy as np
import torch

SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"All seeds fixed to {SEED}")

## 2. Imports & Configuration

In [ ]:
from __future__ import annotations

import gc
import html
import json
import math
import os
import re
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# ── Paths — update these before running ──────────────────────────────────
ROOT = Path(".")  # update to your data directory
INPUT_PATHS = {
    "train":      Path("data/spanish_train.csv"),
    "validation": Path("data/spanish_val.csv"),
    "test":       Path("data/spanish_test.csv"),
}

# ── NEW output directory — does NOT overwrite the original run ─────────────
OUTPUT_DIR                  = ROOT / "outputs" / "spanish_stance_three_lm"
FIGURES_DIR                 = OUTPUT_DIR / "figures"
REPORT_PATH                 = OUTPUT_DIR / "spanish_stance_three_lm_report.md"
SUMMARY_PATH                = OUTPUT_DIR / "summary.json"
VAL_RESULTS_PATH            = OUTPUT_DIR / "validation_results.csv"
TEST_RESULTS_PATH           = OUTPUT_DIR / "test_results.csv"
TEST_CLASSWISE_METRICS_PATH = OUTPUT_DIR / "test_classwise_metrics.csv"
PROFILE_PATH                = OUTPUT_DIR / "dataset_profile.json"
BEST_PREDICTIONS_PATH       = OUTPUT_DIR / "best_supervised_predictions.csv"
THREE_LM_PREDICTIONS_PATH   = OUTPUT_DIR / "three_lm_predictions.csv"
FEATURES_PATH               = OUTPUT_DIR / "best_model_top_features.csv"
ERRORS_PATH                 = OUTPUT_DIR / "best_model_errors.csv"
THREE_LM_ERRORS_PATH        = OUTPUT_DIR / "three_lm_errors.csv"
TRAINING_SUMMARY_PATH       = OUTPUT_DIR / "three_lm_training_summary.json"
THREE_LM_MODEL_DIR               = OUTPUT_DIR / "three_lm_models"
# HF token: loaded automatically from Colab secrets (HF_TOKEN)

# ── Model hyperparameters ──────────────────────────────────────────────────
BASE_MODEL_ID                  = "DeepESP/gpt2-spanish"
LM_MAX_LENGTH                  = 64
LM_BATCH_SIZE                  = 4
LM_EVAL_BATCH_SIZE             = 16
LM_EPOCHS                      = 10
LM_LEARNING_RATE               = 5e-6
LM_WEIGHT_DECAY                = 0.01
LM_MAX_TRAIN_SAMPLES_PER_LABEL = 10000

# ── Label constants & regex ────────────────────────────────────────────────
LABELS       = ["AGAINST", "FAVOR", "NEUTRAL"]
URL_RE        = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE    = re.compile(r"@\w+")
HASHTAG_RE    = re.compile(r"#(\w+)")
RT_RE         = re.compile(r"\brt\b:?", flags=re.IGNORECASE)
WHITESPACE_RE = re.compile(r"\s+")

## 3. Classes & Helper Functions

In [ ]:
@dataclass
class ThreeLMModel:
    """Three-LM argmin classifier: predicts the class whose LM assigns lowest perplexity."""
    tokenizer: AutoTokenizer
    against_model: AutoModelForCausalLM
    favor_model: AutoModelForCausalLM
    neutral_model: AutoModelForCausalLM
    config_name: str
    device: str

    def predict(self, texts) -> tuple[np.ndarray, np.ndarray]:
        docs = list(texts)
        against_scores = score_model_nll(
            self.against_model, self.tokenizer, docs, self.device,
            batch_size=LM_EVAL_BATCH_SIZE, max_length=LM_MAX_LENGTH,
        )
        favor_scores = score_model_nll(
            self.favor_model, self.tokenizer, docs, self.device,
            batch_size=LM_EVAL_BATCH_SIZE, max_length=LM_MAX_LENGTH,
        )
        neutral_scores = score_model_nll(
            self.neutral_model, self.tokenizer, docs, self.device,
            batch_size=LM_EVAL_BATCH_SIZE, max_length=LM_MAX_LENGTH,
        )
        # score_matrix columns: [AGAINST, FAVOR, NEUTRAL]
        score_matrix = np.column_stack([against_scores, favor_scores, neutral_scores])
        # argmin -> index of lowest perplexity -> predicted class
        predicted_indices = np.argmin(score_matrix, axis=1)
        label_array = np.array(LABELS)
        predictions = label_array[predicted_indices]
        perplexities = np.exp(np.clip(score_matrix, -50, 50))
        return predictions, perplexities


class TweetCausalLMDataset(Dataset):
    def __init__(self, texts: list[str], tokenizer: AutoTokenizer, max_length: int) -> None:
        self.records = tokenizer(
            texts, truncation=True, max_length=max_length,
            padding=False, add_special_tokens=True,
        )

    def __len__(self) -> int:
        return len(self.records["input_ids"])

    def __getitem__(self, idx: int) -> dict[str, list[int]]:
        return {
            "input_ids": self.records["input_ids"][idx],
            "attention_mask": self.records["attention_mask"][idx],
        }

In [ ]:
def ensure_dirs() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def normalize_text(text: str) -> str:
    text = html.unescape(str(text))
    text = text.replace("\n", " ").replace("\r", " ")
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(r" \1 ", text)
    text = RT_RE.sub(" ", text)
    text = text.lower()
    text = text.replace("\u2019", "'").replace("`", "'")
    text = WHITESPACE_RE.sub(" ", text).strip()
    return text


def load_split(split_name: str, path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    df["split"] = split_name
    df["tweet_raw"] = df["TWEET"].astype(str)
    df["text_clean"] = df["tweet_raw"].map(normalize_text)
    df["token_count"] = df["text_clean"].str.split().map(len)
    df["char_count"] = df["text_clean"].str.len()
    df["empty_after_cleaning"] = df["text_clean"].eq("")
    return df


def load_data() -> dict[str, pd.DataFrame]:
    return {split: load_split(split, path) for split, path in INPUT_PATHS.items()}


def load_hf_token() -> str:
    try:
        from google.colab import userdata
        colab_token = userdata.get("HF_TOKEN")
        if colab_token:
            return colab_token.strip()
    except Exception:
        pass
    env_token = os.environ.get("HF_TOKEN", "").strip()
    if env_token:
        return env_token
    if HF_ENV_PATH.exists():
        for raw_line in HF_ENV_PATH.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            if key.strip() == "HF_TOKEN":
                token = value.strip().strip('"').strip("'")
                if token:
                    return token
    return ""


def huggingface_kwargs() -> dict[str, object]:
    token = os.environ.get("HF_TOKEN", "").strip()
    kwargs = {}
    if token:
        kwargs["token"] = token
    return kwargs


def resolve_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_built() and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def load_tokenizer(model_source: str) -> AutoTokenizer:
    kwargs = huggingface_kwargs()
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_source, **kwargs)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(model_source, use_fast=False, **kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer


def collate_lm_batch(batch, pad_token_id):
    max_length = max(len(item["input_ids"]) for item in batch)
    input_ids, attention_mask = [], []
    for item in batch:
        pad_length = max_length - len(item["input_ids"])
        input_ids.append(item["input_ids"] + [pad_token_id] * pad_length)
        attention_mask.append(item["attention_mask"] + [0] * pad_length)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }


def score_predictions(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=LABELS, average="macro")),
    }
    for idx, label in enumerate(LABELS):
        lower = label.lower()
        metrics[f"precision_{lower}"] = float(precision[idx])
        metrics[f"recall_{lower}"]    = float(recall[idx])
        metrics[f"f1_{lower}"]        = float(f1[idx])
        metrics[f"support_{lower}"]   = int(support[idx])
    return metrics


def save_fig(fig, filename):
    path = FIGURES_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return filename

## 4. Data Loading & Dataset Profile

In [ ]:
def dataset_profile(splits):
    combined = pd.concat(splits.values(), ignore_index=True)
    profile = {
        "total_rows": int(combined.shape[0]),
        "duplicate_rows_overall": int(combined.duplicated(subset=["id_str", "tweet_raw", "LABEL"]).sum()),
        "duplicate_clean_text_overall": int(combined.duplicated(subset=["text_clean"]).sum()),
        "splits": {},
        "cross_split_overlap_clean_text": {},
    }
    for split_name, df in splits.items():
        label_counts = df["LABEL"].value_counts().reindex(LABELS, fill_value=0)
        profile["splits"][split_name] = {
            "rows": int(df.shape[0]),
            "empty_after_cleaning": int(df["empty_after_cleaning"].sum()),
            "label_counts": {label: int(label_counts[label]) for label in LABELS},
            "avg_token_count": round(float(df["token_count"].mean()), 2),
            "median_token_count": int(df["token_count"].median()),
        }
    split_names = list(splits)
    for idx, left_name in enumerate(split_names):
        left_texts = set(splits[left_name]["text_clean"])
        for right_name in split_names[idx + 1:]:
            overlap = left_texts.intersection(set(splits[right_name]["text_clean"]))
            profile["cross_split_overlap_clean_text"][f"{left_name}_vs_{right_name}"] = int(len(overlap))
    return profile

In [ ]:
ensure_dirs()
splits = load_data()
profile = dataset_profile(splits)
PROFILE_PATH.write_text(json.dumps(profile, indent=2), encoding="utf-8")
print(json.dumps(profile, indent=2))

## 5. Exploratory Plots

In [ ]:
def plot_class_distribution(splits):
    records = []
    for split_name, df in splits.items():
        counts = df["LABEL"].value_counts().reindex(LABELS, fill_value=0)
        for label in LABELS:
            records.append({"split": split_name.title(), "label": label, "count": counts[label]})
    plot_df = pd.DataFrame(records)
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=plot_df, x="split", y="count", hue="label", palette="Set2", ax=ax)
    ax.set_title("CIC-ES Class Distribution by Split")
    ax.set_xlabel("")
    ax.set_ylabel("Tweet count")
    for container in ax.containers:
        ax.bar_label(container, fmt="%d", padding=3, fontsize=9)
    return save_fig(fig, "01_class_distribution.png")


def plot_tweet_length_distribution(splits):
    plot_df = pd.concat(splits.values(), ignore_index=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=plot_df, x="LABEL", y="token_count", order=LABELS,
                color=sns.color_palette("Set2")[0], ax=ax)
    ax.set_title("Cleaned Tweet Length by Label")
    ax.set_xlabel("")
    ax.set_ylabel("Token count")
    return save_fig(fig, "02_token_length_by_label.png")


plot_class_distribution(splits)
plot_tweet_length_distribution(splits)

from IPython.display import Image, display
display(Image(str(FIGURES_DIR / "01_class_distribution.png")))
display(Image(str(FIGURES_DIR / "02_token_length_by_label.png")))

## 6. Supervised Model Selection (TF-IDF + LinearSVC)

In [ ]:
def candidate_models():
    return [
        ("word_tfidf_svc_c0.5", Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.97, sublinear_tf=True)),
            ("clf", LinearSVC(C=0.5)),
        ])),
        ("word_tfidf_svc_c1.0", Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.97, sublinear_tf=True)),
            ("clf", LinearSVC(C=1.0)),
        ])),
        ("char_tfidf_svc_c0.5", Pipeline([
            ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
            ("clf", LinearSVC(C=0.5)),
        ])),
        ("char_tfidf_svc_c1.0", Pipeline([
            ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
            ("clf", LinearSVC(C=1.0)),
        ])),
        ("hybrid_tfidf_svc_c0.5", Pipeline([
            ("features", FeatureUnion([
                ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.97, sublinear_tf=True)),
                ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
            ])),
            ("clf", LinearSVC(C=0.5)),
        ])),
        ("hybrid_tfidf_svc_c1.0", Pipeline([
            ("features", FeatureUnion([
                ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.97, sublinear_tf=True)),
                ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
            ])),
            ("clf", LinearSVC(C=1.0)),
        ])),
    ]
def select_supervised_model(train_df, validation_df):
    records, best_name, best_model, best_score = [], "", None, -1.0
    for name, model in candidate_models():
        fitted = clone(model)
        fitted.fit(train_df["text_clean"], train_df["LABEL"])
        val_predictions = fitted.predict(validation_df["text_clean"])
        metrics = score_predictions(validation_df["LABEL"], val_predictions)
        metrics["model"] = name
        metrics["polar_f1avg"] = (metrics["f1_against"] + metrics["f1_favor"]) / 2
        records.append(metrics)
        if metrics["polar_f1avg"] > best_score:
            best_score = metrics["polar_f1avg"]
            best_name  = name
            best_model = fitted
    assert best_model is not None
    results = pd.DataFrame(records).sort_values("polar_f1avg", ascending=False)
    results.to_csv(VAL_RESULTS_PATH, index=False)
    print(f"Selected supervised model: {best_name} (validation polar_f1avg={best_score:.4f})")
    return best_model, results


print("Selecting supervised baseline", flush=True)
supervised_model, validation_results = select_supervised_model(splits["train"], splits["validation"])
validation_results[["model", "polar_f1avg", "macro_f1", "f1_against", "f1_favor", "f1_neutral"]].head(6)

## 7. Three-LM Model — One GPT-2 per Class

Instead of training only on polar classes and using a tau threshold to abstain into NEUTRAL, we now train **three separate language models** — one per class — and classify each tweet by argmin perplexity. This directly addresses the low NEUTRAL F1 of the two-LM approach, since NEUTRAL tweets are expected to have a genuinely distinct token distribution (generic Spanish Twitter language, unrelated to the independence debate).

In [ ]:
def train_causal_lm_for_label(
    label: str, texts: list[str], output_dir: Path,
    tokenizer: AutoTokenizer, device: str,
    training_summary: dict,
) -> AutoModelForCausalLM:
    output_dir.mkdir(parents=True, exist_ok=True)
    existing_weights = (
        (output_dir / "model.safetensors").exists() or
        (output_dir / "pytorch_model.bin").exists()
    )
    if existing_weights:
        print(f"Reusing cached three-LM model for {label} from {output_dir}", flush=True)
        training_summary[label] = {"reused_from_disk": True, "model_dir": str(output_dir)}
        model = AutoModelForCausalLM.from_pretrained(str(output_dir), torch_dtype=torch.float32)
        model.to(device)
        model.eval()
        return model

    texts = [t for t in texts if len(t.strip()) > 5][:LM_MAX_TRAIN_SAMPLES_PER_LABEL]
    print(f"Training three-LM model for {label} on {len(texts)} tweets", flush=True)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.float32, **huggingface_kwargs())
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    model.to(device)
    model.train()

    dataset    = TweetCausalLMDataset(texts, tokenizer, LM_MAX_LENGTH)
    g = torch.Generator()
    g.manual_seed(SEED)
    dataloader = DataLoader(
        dataset, batch_size=LM_BATCH_SIZE, shuffle=True, generator=g,
        collate_fn=lambda batch: collate_lm_batch(batch, tokenizer.pad_token_id),
    )
    optimizer = AdamW(model.parameters(), lr=LM_LEARNING_RATE, weight_decay=LM_WEIGHT_DECAY)

    history = []
    for epoch in range(LM_EPOCHS):
        running_loss, valid_steps = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(dataloader, start=1):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_t       = input_ids.clone()
            labels_t[attention_mask == 0] = -100

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels_t)

            if not torch.isfinite(outputs.loss):
                print(f"Warning: non-finite loss at step {step}, skipping")
                optimizer.zero_grad(set_to_none=True)
                continue

            outputs.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            running_loss += float(outputs.loss.item())
            valid_steps  += 1

            if step % 50 == 0 or step == len(dataloader):
                mean = running_loss / valid_steps if valid_steps else float("nan")
                print(f"{label} epoch {epoch+1} step {step}/{len(dataloader)} mean_loss={mean:.4f}", flush=True)

        epoch_loss = running_loss / max(valid_steps, 1)
        history.append({"epoch": float(epoch + 1), "loss": epoch_loss})
        print(f"{label} epoch {epoch+1}/{LM_EPOCHS} loss={epoch_loss:.4f}", flush=True)

    model.config.tie_word_embeddings = False
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    model.eval()
    training_summary[label] = {
        "tweets": len(texts), "epochs": LM_EPOCHS, "batch_size": LM_BATCH_SIZE,
        "learning_rate": LM_LEARNING_RATE, "max_length": LM_MAX_LENGTH,
        "device": device, "history": history, "model_dir": str(output_dir),
    }
    return model


def score_model_nll(model, tokenizer, texts, device, batch_size, max_length):
    model.eval()
    scores = []
    total_batches = math.ceil(len(texts) / batch_size)
    with torch.no_grad():
        for batch_index, start in enumerate(range(0, len(texts), batch_size), start=1):
            batch_texts = texts[start : start + batch_size]
            encodings   = tokenizer(
                batch_texts, return_tensors="pt", padding=True,
                truncation=True, max_length=max_length,
            )
            input_ids      = encodings["input_ids"].to(device)
            attention_mask = encodings["attention_mask"].to(device)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            shift_logits   = outputs.logits[:, :-1, :].contiguous()
            shift_labels   = input_ids[:, 1:].contiguous()
            shift_mask     = attention_mask[:, 1:].contiguous()
            token_losses   = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1), reduction="none",
            ).view(shift_labels.size())
            token_losses     = token_losses * shift_mask
            seq_token_counts = shift_mask.sum(dim=1).clamp(min=1)
            seq_scores       = token_losses.sum(dim=1) / seq_token_counts
            scores.append(seq_scores.detach().cpu().numpy())
            if batch_index % 20 == 0 or batch_index == total_batches:
                print(f"Scoring batches {batch_index}/{total_batches} on {device}", flush=True)
    return np.concatenate(scores, axis=0)

In [ ]:
def train_three_lm_model(train_df, validation_df):
    device    = resolve_device()
    tokenizer = load_tokenizer(BASE_MODEL_ID)
    training_summary = {"base_model_id": BASE_MODEL_ID, "device": device,
                        "design": "Three-LM argmin: one GPT-2 per class, classify by lowest perplexity."}

    against_model = train_causal_lm_for_label(
        "AGAINST",
        train_df.loc[train_df["LABEL"] == "AGAINST", "text_clean"].tolist(),
        THREE_LM_MODEL_DIR / "against", tokenizer, device, training_summary,
    )
    favor_model = train_causal_lm_for_label(
        "FAVOR",
        train_df.loc[train_df["LABEL"] == "FAVOR", "text_clean"].tolist(),
        THREE_LM_MODEL_DIR / "favor", tokenizer, device, training_summary,
    )
    neutral_model = train_causal_lm_for_label(
        "NEUTRAL",
        train_df.loc[train_df["LABEL"] == "NEUTRAL", "text_clean"].tolist(),
        THREE_LM_MODEL_DIR / "neutral", tokenizer, device, training_summary,
    )

    three_lm = ThreeLMModel(
        tokenizer=tokenizer,
        against_model=against_model,
        favor_model=favor_model,
        neutral_model=neutral_model,
        config_name="deepesp_gpt2_three_lm",
        device=device,
    )

    # Evaluate on validation to report
    print("Scoring validation tweets with three-LM model", flush=True)
    val_predictions, val_perplexities = three_lm.predict(validation_df["text_clean"])
    val_metrics = score_predictions(validation_df["LABEL"], val_predictions)
    val_metrics["model"] = three_lm.config_name
    val_metrics["polar_f1avg"] = (val_metrics["f1_against"] + val_metrics["f1_favor"]) / 2
    val_results = pd.DataFrame([val_metrics])

    TRAINING_SUMMARY_PATH.write_text(json.dumps(training_summary, indent=2), encoding="utf-8")
    print(
        f"Three-LM validation polar_f1avg={val_metrics['polar_f1avg']:.4f} "
        f"| macro_f1={val_metrics['macro_f1']:.4f} "
        f"| f1_against={val_metrics['f1_against']:.4f} "
        f"| f1_favor={val_metrics['f1_favor']:.4f} "
        f"| f1_neutral={val_metrics['f1_neutral']:.4f}"
    )
    return three_lm, val_results


print("Training three-LM model (AGAINST + FAVOR + NEUTRAL)", flush=True)
three_lm_model, three_lm_val_results = train_three_lm_model(splits["train"], splits["validation"])
three_lm_val_results[["model", "polar_f1avg", "macro_f1", "f1_against", "f1_favor", "f1_neutral"]]

## 8. Test Evaluation & Visualisations

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, filename):
    matrix = confusion_matrix(y_true, y_pred, labels=LABELS)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    return save_fig(fig, filename)


def plot_perplexity_distribution(perplexities, labels_series, filename):
    """Show per-class perplexity distributions for each of the three LMs."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
    colors = sns.color_palette("Set2", n_colors=3)
    lm_names = ["AGAINST LM", "FAVOR LM", "NEUTRAL LM"]
    for col_idx, (ax, lm_name) in enumerate(zip(axes, lm_names)):
        for label_idx, (label, color) in enumerate(zip(LABELS, colors)):
            mask   = labels_series == label
            values = np.log1p(perplexities[mask, col_idx])
            ax.hist(values, bins=40, density=True, alpha=0.45, color=color, label=label)
        ax.set_title(f"log-perplexity under {lm_name}")
        ax.set_xlabel("log(1 + perplexity)")
        ax.legend(fontsize=8)
    fig.suptitle("Validation Perplexity Distributions by True Label", fontsize=13)
    return save_fig(fig, filename)


def build_classwise_metric_frame(test_results):
    pretty = {"best_supervised": "Best Supervised", "deepesp_gpt2_three_lm": "GPT-2 Three-LM"}
    records = []
    for _, row in test_results.iterrows():
        model_name = pretty.get(row["model"], row["model"])
        for label in LABELS:
            lower = label.lower()
            for metric in ["precision", "recall", "f1"]:
                records.append({"model": model_name, "label": label,
                                "metric": metric.upper(), "value": float(row[f"{metric}_{lower}"])})
    return pd.DataFrame(records)


def plot_classwise_metric_dashboard(metrics_df, filename):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
    palette   = {"Best Supervised": "#2a9d8f", "GPT-2 Three-LM": "#e76f51"}
    for ax, metric_name in zip(axes, ["PRECISION", "RECALL", "F1"]):
        subset = metrics_df.loc[metrics_df["metric"] == metric_name]
        sns.barplot(data=subset, x="label", y="value", hue="model",
                    order=LABELS, hue_order=list(palette.keys()), palette=palette, ax=ax)
        ax.set_ylim(0, 1)
        ax.set_xlabel("")
        ax.set_ylabel("Score")
        ax.set_title(f"Test {metric_name} by Class")
        for container in ax.containers:
            ax.bar_label(container, fmt="%.2f", padding=3, fontsize=8)
    axes[0].legend(title="")
    for ax in axes[1:]:
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()
    return save_fig(fig, filename)


def plot_classwise_metric_heatmap(metrics_df, filename):
    heatmap_df = (
        metrics_df.assign(column=lambda d: d["metric"] + "\n" + d["label"])
        .pivot(index="model", columns="column", values="value")
        .reindex(index=["Best Supervised", "GPT-2 Three-LM"])
    )
    ordered_columns = [f"{m}\n{l}" for m in ["PRECISION", "RECALL", "F1"] for l in LABELS]
    heatmap_df = heatmap_df.reindex(columns=ordered_columns)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap="YlGnBu",
                vmin=0, vmax=1, cbar_kws={"label": "Score"}, ax=ax)
    ax.set_title("Test Precision, Recall, and F1 Heatmap")
    ax.set_xlabel("")
    ax.set_ylabel("")
    return save_fig(fig, filename)


def plot_overall_metric_comparison(test_results, filename):
    pretty = {"best_supervised": "Best Supervised", "deepesp_gpt2_three_lm": "GPT-2 Three-LM"}
    plot_df = pd.DataFrame([
        {"model": pretty.get(row["model"], row["model"]), "metric": m, "value": float(row[col])}
        for _, row in test_results.iterrows()
        for m, col in [("F1avg", "polar_f1avg"), ("Macro F1", "macro_f1")]
    ])
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=plot_df, x="metric", y="value", hue="model",
                hue_order=["Best Supervised", "GPT-2 Three-LM"],
                palette=["#2a9d8f", "#e76f51"], ax=ax)
    ax.set_ylim(0, 1)
    ax.set_xlabel("")
    ax.set_ylabel("Score")
    ax.set_title("Overall Test Metric Comparison")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.4f", padding=3, fontsize=9)
    ax.legend(title="")
    return save_fig(fig, filename)

In [ ]:
def extract_top_features(model, top_n=20):
    if "features" in model.named_steps:
        feature_names = model.named_steps["features"].get_feature_names_out()
    else:
        feature_names = model.named_steps["tfidf"].get_feature_names_out()
    classifier = model.named_steps["clf"]
    top_rows = []
    for class_index, label in enumerate(classifier.classes_):
        coef         = classifier.coef_[class_index]
        positive_idx = np.argsort(coef)[-top_n:][::-1]
        negative_idx = np.argsort(coef)[:top_n]
        for rank, idx in enumerate(positive_idx, start=1):
            top_rows.append({"class_label": label, "direction": "positive",
                             "rank": rank, "feature": feature_names[idx], "weight": float(coef[idx])})
        for rank, idx in enumerate(negative_idx, start=1):
            top_rows.append({"class_label": label, "direction": "negative",
                             "rank": rank, "feature": feature_names[idx], "weight": float(coef[idx])})
    features = pd.DataFrame(top_rows)
    features.to_csv(FEATURES_PATH, index=False)
    return features


def build_prediction_frame(df, predictions, extra_columns=None):
    pred_df = df[["id_str", "split", "LABEL", "tweet_raw", "text_clean", "token_count", "char_count"]].copy()
    pred_df["prediction"] = predictions
    pred_df["is_correct"] = pred_df["LABEL"] == pred_df["prediction"]
    if extra_columns:
        for key, values in extra_columns.items():
            pred_df[key] = values
    return pred_df

In [ ]:
def evaluate_on_test(supervised_model, three_lm, test_df, validation_df):
    # Supervised
    supervised_predictions = supervised_model.predict(test_df["text_clean"])
    supervised_metrics     = score_predictions(test_df["LABEL"], supervised_predictions)
    supervised_metrics["model"] = "best_supervised"

    # Three-LM
    print("Scoring test tweets with three-LM model", flush=True)
    three_lm_predictions, three_lm_perplexities = three_lm.predict(test_df["text_clean"])
    three_lm_metrics          = score_predictions(test_df["LABEL"], three_lm_predictions)
    three_lm_metrics["model"] = three_lm.config_name

    test_results = pd.DataFrame([supervised_metrics, three_lm_metrics])
    test_results["polar_f1avg"] = (test_results["f1_against"] + test_results["f1_favor"]) / 2
    test_results.to_csv(TEST_RESULTS_PATH, index=False)

    # Save predictions
    best_pred_df = build_prediction_frame(test_df, supervised_predictions)
    best_pred_df.to_csv(BEST_PREDICTIONS_PATH, index=False)
    best_pred_df.loc[~best_pred_df["is_correct"]].head(100).to_csv(ERRORS_PATH, index=False)

    three_lm_extra = {
        "ppl_against": three_lm_perplexities[:, 0],
        "ppl_favor":   three_lm_perplexities[:, 1],
        "ppl_neutral": three_lm_perplexities[:, 2],
    }
    three_lm_pred_df = build_prediction_frame(test_df, three_lm_predictions, three_lm_extra)
    three_lm_pred_df.to_csv(THREE_LM_PREDICTIONS_PATH, index=False)
    three_lm_pred_df.loc[~three_lm_pred_df["is_correct"]].head(100).to_csv(THREE_LM_ERRORS_PATH, index=False)

    # Confusion matrices
    confusion_supervised = plot_confusion_matrix(
        test_df["LABEL"], supervised_predictions,
        "Best Supervised Model: Test Confusion Matrix", "03_best_supervised_confusion_matrix.png",
    )
    confusion_three_lm = plot_confusion_matrix(
        test_df["LABEL"], three_lm_predictions,
        "GPT-2 Three-LM: Test Confusion Matrix", "04_three_lm_confusion_matrix.png",
    )

    # Perplexity distribution plot (on validation for interpretability)
    print("Scoring validation tweets for perplexity plot", flush=True)
    _, val_perplexities = three_lm.predict(validation_df["text_clean"])
    perplexity_plot = plot_perplexity_distribution(
        val_perplexities, validation_df["LABEL"].reset_index(drop=True),
        "05_validation_perplexity_distribution.png",
    )

    # Classwise metrics
    classwise_metrics = build_classwise_metric_frame(test_results)
    classwise_metrics.to_csv(TEST_CLASSWISE_METRICS_PATH, index=False)
    metric_dashboard    = plot_classwise_metric_dashboard(classwise_metrics, "06_test_precision_recall_f1_dashboard.png")
    metric_heatmap      = plot_classwise_metric_heatmap(classwise_metrics, "07_test_classwise_metric_heatmap.png")
    overall_metric_plot = plot_overall_metric_comparison(test_results, "08_test_overall_metric_comparison.png")

    artifacts = {
        "supervised_confusion_matrix": confusion_supervised,
        "three_lm_confusion_matrix":   confusion_three_lm,
        "perplexity_distribution":     perplexity_plot,
        "classwise_metric_dashboard":  metric_dashboard,
        "classwise_metric_heatmap":    metric_heatmap,
        "overall_metric_comparison":   overall_metric_plot,
    }
    return test_results, artifacts


print("Evaluating on test split", flush=True)
test_results, artifacts = evaluate_on_test(
    supervised_model, three_lm_model, splits["test"], splits["validation"]
)
top_features = extract_top_features(supervised_model)
test_results["polar_f1avg"] = (test_results["f1_against"] + test_results["f1_favor"]) / 2
test_results[["model", "polar_f1avg", "macro_f1", "accuracy", "f1_against", "f1_favor", "f1_neutral"]]

In [ ]:
from IPython.display import Image, display
for fname in [
    "03_best_supervised_confusion_matrix.png",
    "04_three_lm_confusion_matrix.png",
    "05_validation_perplexity_distribution.png",
    "06_test_precision_recall_f1_dashboard.png",
    "07_test_classwise_metric_heatmap.png",
    "08_test_overall_metric_comparison.png",
]:
    print(fname)
    display(Image(str(FIGURES_DIR / fname)))

## 9. Write Report & Summary

In [ ]:
def render_table(df, columns):
    disp = df.loc[:, columns].copy()
    for col in disp.columns:
        if disp[col].dtype.kind in {"f"}:
            disp[col] = disp[col].map(lambda x: f"{x:.4f}")
    headers = list(disp.columns)
    rows    = [headers] + disp.astype(str).values.tolist()
    widths  = [max(len(row[i]) for row in rows) for i in range(len(headers))]
    def fmt(row):
        return "| " + " | ".join(v.ljust(widths[i]) for i, v in enumerate(row)) + " |"
    separator = "| " + " | ".join("-" * w for w in widths) + " |"
    body = [fmt(headers), separator]
    body.extend(fmt(row) for row in disp.astype(str).values.tolist())
    return "\n".join(body)

def write_report(profile, validation_results, three_lm_val_results, test_results, artifacts, top_features):
    best_validation = validation_results.iloc[0]
    best_test       = test_results.iloc[0]
    three_lm_test   = test_results.iloc[1]
    mbart_reference_polar  = 0.7472
    xlmr_reference_polar   = 0.7357
    tfidf_reference_polar  = 0.7109

    report = f"""# Spanish Stance Detection — Three-LM Variant

## Design
This run replaces the two-LM tau-abstention approach with a three-LM argmin design.
One GPT-2 language model is fine-tuned per class (AGAINST, FAVOR, NEUTRAL).
A tweet is assigned to whichever model assigns it the lowest average token-level negative log-likelihood.
No abstention threshold is needed.

## Dataset profile
- Train rows: {profile["splits"]["train"]["rows"]}, validation rows: {profile["splits"]["validation"]["rows"]}, test rows: {profile["splits"]["test"]["rows"]}.
- Train class counts: {profile["splits"]["train"]["label_counts"]}.

## Supervised baseline

{render_table(validation_results.head(6), ['model', 'polar_f1avg', 'macro_f1', 'f1_against', 'f1_favor', 'f1_neutral'])}

Best supervised model: `{best_validation['model']}` with validation polar F1avg `{best_validation['polar_f1avg']:.4f}`.

## Three-LM validation results

{render_table(three_lm_val_results, ['model', 'polar_f1avg', 'macro_f1', 'f1_against', 'f1_favor', 'f1_neutral'])}

## Final test results

{render_table(test_results, ['model', 'polar_f1avg', 'macro_f1', 'f1_against', 'f1_favor', 'f1_neutral'])}

Primary metric (polar F1avg, following Zotova et al.):
- Three-LM: `{three_lm_test['polar_f1avg']:.4f}`
- Supervised TF-IDF: `{best_test['polar_f1avg']:.4f}`
- mBERT (Zotova et al.): `{mbart_reference_polar:.4f}`
- XLM-R (Zotova et al.): `{xlmr_reference_polar:.4f}`
- TF-IDF+SVM (Zotova et al.): `{tfidf_reference_polar:.4f}`

## Output directory
All outputs written to: `{OUTPUT_DIR}`
Original two-LM results preserved in: `{ROOT / 'outputs' / 'spanish_stance_project'}`
"""
    REPORT_PATH.write_text(report, encoding="utf-8")
    print(f"Report written to {REPORT_PATH}")


summary = {
    "best_supervised_model":                validation_results.iloc[0]["model"],
    "best_supervised_test_polar_f1avg":     float(test_results.iloc[0]["polar_f1avg"]),
    "best_supervised_test_macro_f1":        float(test_results.iloc[0]["macro_f1"]),
    "three_lm_model":                       three_lm_model.config_name,
    "three_lm_validation_polar_f1avg":      float(three_lm_val_results.iloc[0]["polar_f1avg"]),
    "three_lm_validation_macro_f1":         float(three_lm_val_results.iloc[0]["macro_f1"]),
    "three_lm_test_polar_f1avg":            float(test_results.iloc[1]["polar_f1avg"]),
    "three_lm_test_macro_f1":               float(test_results.iloc[1]["macro_f1"]),
    "three_lm_test_f1_against":             float(test_results.iloc[1]["f1_against"]),
    "three_lm_test_f1_favor":               float(test_results.iloc[1]["f1_favor"]),
    "three_lm_test_f1_neutral":             float(test_results.iloc[1]["f1_neutral"]),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")

write_report(
    profile, validation_results, three_lm_val_results,
    test_results, artifacts, top_features,
)
print(json.dumps(summary, indent=2))